# bge-m3
Embed the legal chunks with one model and score it on `eval_questions.json`.
To test another model, copy this notebook and change only the config cell.

In [1]:
import sys
sys.path.insert(0, "..")  # common.py lives one folder up
import pandas as pd
import common

# ---- config (the only cell that differs between model notebooks) ----
NAME = "bge-m3"                 
MODEL_NAME = "BAAI/bge-m3"
QUERY_PREFIX = ""              
PASSAGE_PREFIX = ""             
MAX_LENGTH = 1024
BATCH_SIZE = 8
DEVICE = "cpu"                 # None = auto; MPS ran out of memory on this machine
FORCE = False                   # True = ignore the cache and re-embed

## 1. Chunks

In [2]:
articles = common.load_articles()
chunks = common.build_chunks(articles)
questions = common.load_questions()
print(f"{len(articles)} articles -> {len(chunks)} chunks, {len(questions)} questions")
print("chunking:", common.CHUNKING_VERSION)
print("longest chunk (chars):", max(len(c["embed_text"]) for c in chunks))
print(chunks[0]["embed_text"])

92 articles -> 115 chunks, 34 questions
chunking: v1-article-maxchars1500
longest chunk (chars): 1578
ជំពូកទី១ បទប្បញ្ញត្តិទូទៅ · មាត្រា១ គោលបំណង
ច្បាប់នេះមានគោលបំណងធានាសុវត្ថិភាព សណ្តាប់ធ្នាប់ចរាចរណ៍ផ្លូវគោក និងធានាការពារសុខភាពនិងអាយុជីវិតមនុស្ស សត្វ ទ្រព្យសម្បត្តិ និងបរិស្ថាន។


## 2. Embed + evaluate
Vectors are cached in this folder (`vectors.npy` + `chunks.json` + `info.json`), so re-running is fast.

In [3]:
result = common.run_experiment(
    NAME, MODEL_NAME, chunks, questions,
    query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE, device=DEVICE, force=FORCE,
)
print(result["info"])
result["metrics"]

/Users/aester/School/khmer-doc/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 15/15 [02:23<00:00,  9.60s/it]


[saved] (115, 1024) in 144.0s -> /Users/aester/School/khmer-doc/app/engine/model/bge-m3
{'name': 'bge-m3', 'model_name': 'BAAI/bge-m3', 'dim': 1024, 'n_chunks': 115, 'max_length': 1024, 'passage_prefix': '', 'chunking_version': 'v1-article-maxchars1500', 'content_hash': '566159e4e5a70ba1', 'embed_seconds': 143.99, 'created': '2026-11-10T19:58:01'}


{'recall@1': 0.9118, 'recall@3': 0.9118, 'recall@5': 0.9706, 'mrr': 0.9314}

## 3. Misses
Questions where no relevant article is in the top 5 (or ranked low). Check whether the question or the label is wrong before blaming the model.

In [4]:
df = pd.DataFrame(result["details"])
df[df["rank"].isna() | (df["rank"] > 1)][["id", "question", "relevant", "rank", "top5"]]

,id,question,relevant,rank,top5
10,q11,តើអនុញ្ញាតឲ្យប្រើទូរស័ព្ទនៅពេលកំពុងបើកបរដែរឬទេ?,[article-8-new],4,"[article-28, article-4, article-27, article-8-..."
19,q20,តើអ្នកបើកបរដែលបើកយឺតត្រូវបើកបរនៅផ្នែកណានៃផ្លូវ?,[article-16],4,"[article-9, article-12, article-27, article-16..."
29,q30,តើអ្នកបើកបរត្រូវប្រកាន់ផ្លូវខាងណា?,[article-7],6,"[article-9, article-10, article-19, article-12..."
